# Causal Inference in Practice
## Week 4 — Causal Graphs (DAGs) · Practice Notebook

> **Block I — Foundations**
>
> A visual algebra for assumptions: read confounders, colliders, and mediators directly off the graph.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · A graph you can simulate

A DAG is just a set of structural equations: each variable is a function of its parents plus noise. Because we *write* those equations, we know the true graph exactly — so we can check what d-separation predicts against what the data actually do.

Throughout we use a simple test: **X and Y are (conditionally) independent iff the coefficient on X, when we regress Y on X (and the conditioning set), is ≈ 0.** A near-zero slope ⇒ independence; a clearly non-zero slope ⇒ dependence.

In [ ]:
import statsmodels.api as sm

def slope(y, *cols):
    """OLS slope on the FIRST regressor, adjusting for the rest.
    Returns (coef, p_value) — coef≈0 ⇔ (conditional) independence."""
    Xmat = sm.add_constant(np.column_stack(cols))
    fit = sm.OLS(np.asarray(y), Xmat).fit()
    return fit.params[1], fit.pvalues[1]

n = 10_000
print('Helper ready. We will read off (coef, p) to judge independence.')

## 2 · The fork (confounder):  X ← Z → Y

Z is a common cause of X and Y, with **no** arrow X → Y. d-separation says the fork is OPEN marginally (so X and Y look associated) but BLOCKED given Z (so `X ⊥ Y | Z`). Adjusting for Z therefore removes the confounding entirely.

In [ ]:
# Fork: Z -> X, Z -> Y, and NO direct X->Y edge (true effect = 0).
Z = RNG.normal(size=n)
Xf = 1.0*Z + RNG.normal(size=n)
Yf = 2.0*Z + RNG.normal(size=n)        # depends on Z only, not on X

c_marg, p_marg = slope(Yf, Xf)         # X alone
c_cond, p_cond = slope(Yf, Xf, Z)      # X adjusting for Z
print(f'marginal   X->Y slope = {c_marg:+.3f}  (p={p_marg:.1e})  '
      '<- spurious, fork is OPEN')
print(f'given Z    X->Y slope = {c_cond:+.3f}  (p={p_cond:.2f})  '
      '<- ~0, fork BLOCKED by Z')
print('\nTrue X->Y effect is 0. Adjusting for Z recovers it.')
assert abs(c_marg) > 0.3,  'fork should look associated marginally'
assert abs(c_cond) < 0.1,  'conditioning on Z should give independence'

The marginal slope is large and the conditional slope collapses to ~0: exactly `X ⊥ Y | Z`. **This is why we adjust for confounders** — conditioning on the fork's middle node blocks the back-door path.

## 3 · The chain (mediator):  X → M → Y

Now X causes Y, but only *through* M. d-separation says the chain is OPEN marginally (X and Y associated) and BLOCKED given M (`X ⊥ Y | M`). Conditioning on a mediator removes the very effect we usually want — the mirror image of the fork's lesson.

In [ ]:
# Chain: X -> M -> Y. The TOTAL effect of X on Y is 1.0*1.0 = 1.0.
Xc = RNG.normal(size=n)
M  = 1.0*Xc + RNG.normal(size=n)
Yc = 1.0*M  + RNG.normal(size=n)       # Y depends on X only via M

c_tot,  p_tot  = slope(Yc, Xc)         # total effect
c_cond, p_cond = slope(Yc, Xc, M)      # adjusting for the mediator
print(f'marginal   X->Y slope = {c_tot:+.3f}  (p={p_tot:.1e})  '
      '<- total effect ~1.0, chain OPEN')
print(f'given M    X->Y slope = {c_cond:+.3f}  (p={p_cond:.2f})  '
      '<- ~0, chain BLOCKED by M')
assert abs(c_tot - 1.0) < 0.1, 'total effect should be ~1.0'
assert abs(c_cond) < 0.1,      'conditioning on M should block the path'

Same arithmetic as the fork — conditioning on the middle node blocks the path — but here the path was the *causal* one. Adjusting for a mediator wrongly zeros out a real effect. The graph, not the regression, tells you which middle node is which.

## 4 · The collider:  X → K ← Y  (the rule reverses)

K is a common effect of X and Y, which are otherwise independent. d-separation says the collider is BLOCKED marginally (`X ⊥ Y`) but OPENS when we condition on K. So conditioning *creates* an association that was not there — collider bias.

In [ ]:
# Collider: X and Y independent; both cause K.
Xk = RNG.normal(size=n)
Yk = RNG.normal(size=n)                # independent of X by construction
K  = 1.0*Xk + 1.0*Yk + RNG.normal(size=n)

c_marg, p_marg = slope(Yk, Xk)         # X alone
c_cond, p_cond = slope(Yk, Xk, K)      # conditioning on the collider
print(f'marginal   X->Y slope = {c_marg:+.3f}  (p={p_marg:.2f})  '
      '<- ~0, collider BLOCKED (true independence)')
print(f'given K    X->Y slope = {c_cond:+.3f}  (p={p_cond:.1e})  '
      '<- non-zero! conditioning OPENED the path')
assert abs(c_marg) < 0.05,            'X and Y are truly independent'
assert c_cond < -0.2,                 'conditioning on K induces (negative) assoc.'

Marginally `X ⊥ Y` (slope ≈ 0); conditioning on K induces a strong **negative** association out of thin air. This is the asymmetry that makes 'control for everything' dangerous: the collider's rule is the reverse of the chain's and fork's.

### 🔧 Exercise 4.1 — open a collider through its DESCENDANT

d-separation says conditioning on a *descendant* of a collider also opens it. Extend the collider world with `D = K + noise` (so D is a child of K), then show that conditioning on **D** — not K itself — still induces an X–Y association.

Fill in the `# TODO`s. The skeleton runs as-is (it just prints `None`s) until you complete it.

In [ ]:
# TODO: build D as a descendant of the collider K, then condition on D.
D = ...        # TODO: D = 1.0*K + RNG.normal(size=n)
c_d = None     # TODO: c_d, _ = slope(Yk, Xk, D)
# print(f'given D (descendant of K): X->Y slope = {c_d:+.3f}')

### ✅ Solution 4.1

In [ ]:
D = 1.0*K + RNG.normal(size=n)        # D is a child of the collider K
c_d, p_d = slope(Yk, Xk, D)
print(f'given D (descendant of K): X->Y slope = {c_d:+.3f}  (p={p_d:.1e})')
print('Conditioning on a descendant of a collider opens it too.')
assert abs(c_d) > 0.1, 'descendant of a collider should still open the path'

## 5 · Back-door adjustment vs. over-adjustment

Now a realistic graph with a **real** effect to recover and two ways to get it wrong. Confounder Z opens a back-door; collider K tempts you to over-adjust.

```
  Z → X ,  Z → Y        (back-door fork — must close)
  X → Y                 (TRUE causal effect = 1.5)
  X → K ← Y             (collider — must NOT touch)
```
The back-door criterion says: adjust for **{Z}**, and for nothing else.

In [ ]:
TRUE = 1.5
Z = RNG.normal(size=n)
X = 1.0*Z + RNG.normal(size=n)
Y = TRUE*X + 2.0*Z + RNG.normal(size=n)     # true X->Y = 1.5
K = 1.0*X + 1.0*Y + RNG.normal(size=n)       # collider (common effect)

naive,   _ = slope(Y, X)              # back-door left OPEN
correct, _ = slope(Y, X, Z)           # back-door CLOSED (valid set {Z})
over,    _ = slope(Y, X, Z, K)        # also conditioning on collider K

print(f'TRUE effect            = {TRUE:.3f}')
print(f'naive  (no adjust)     = {naive:.3f}   <- confounded, too high')
print(f'adjust {{Z}}            = {correct:.3f}   <- recovers the truth')
print(f'adjust {{Z, K}}         = {over:.3f}   <- collider bias, wrong again')
assert abs(correct - TRUE) < 0.1, 'back-door set {Z} should recover 1.5'
assert abs(over - TRUE)    > 0.1, 'adding the collider K should bias it'

Three estimates from one dataset: only the **back-door set {Z}** lands on 1.5. Leaving Z out keeps confounding; adding the collider K *introduces* new bias. The valid adjustment set is the Goldilocks choice the DAG hands you — neither too few controls nor too many.

### 🔧 Exercise 5.1 — find the valid set yourself

New graph: two confounders `A` and `B` both cause `X` and `Y`; the true effect of `X` on `Y` is **0.8**; there is also a mediator `X → Mp → Y` and a collider `X → C ← Y`. Estimate X → Y four ways and decide which adjustment set is valid.

Complete the `# TODO`s so all four estimates print.

In [ ]:
TRUE2 = 0.8
A = RNG.normal(size=n)
B = RNG.normal(size=n)
X2 = 0.7*A + 0.7*B + RNG.normal(size=n)
# Y2 below uses X2, A, B; build it so the DIRECT effect of X2 is 0.8:
Mp = 0.0*X2 + RNG.normal(size=n)             # (kept off the path for simplicity)
Y2 = TRUE2*X2 + 1.0*A + 1.0*B + RNG.normal(size=n)
Cc = 1.0*X2 + 1.0*Y2 + RNG.normal(size=n)    # collider

est_none = None   # TODO: slope(Y2, X2)
est_AB   = None   # TODO: slope(Y2, X2, A, B)
est_ABC  = None   # TODO: slope(Y2, X2, A, B, Cc)   # adds the collider
# print(est_none, est_AB, est_ABC)

### ✅ Solution 5.1

The valid set is **{A, B}** (both confounders, no collider, no mediator). Adjusting for the collider `Cc` on top of {A, B} re-introduces bias.

In [ ]:
est_none, _ = slope(Y2, X2)
est_AB,   _ = slope(Y2, X2, A, B)
est_ABC,  _ = slope(Y2, X2, A, B, Cc)
print(f'TRUE              = {TRUE2:.3f}')
print(f'no adjustment     = {est_none:.3f}   (confounded)')
print(f'adjust {{A, B}}     = {est_AB:.3f}   (valid -> recovers truth)')
print(f'adjust {{A,B,C}}    = {est_ABC:.3f}   (collider bias)')
assert abs(est_AB - TRUE2) < 0.1, 'valid set {A,B} should recover 0.8'
assert abs(est_ABC - TRUE2) > 0.05, 'adding collider C should bias the estimate'

## 6 · Falsifying a DAG with its testable implication

A DAG predicts conditional independencies. Take the chain `Z → X → Y` (no direct Z→Y edge): it implies **`Z ⊥ Y | X`**. We simulate two worlds — one that obeys the DAG and one with a sneaky extra `Z → Y` edge — and let the test tell them apart.

In [ ]:
# World A obeys the DAG: Z -> X -> Y, no direct Z->Y.
Zc  = RNG.normal(size=n)
Xc2 = 1.0*Zc + RNG.normal(size=n)
Ya  = 1.0*Xc2 + RNG.normal(size=n)            # no direct Z effect

# World B violates it: a hidden direct Z -> Y edge sneaks in.
Yb  = 1.0*Xc2 + 0.8*Zc + RNG.normal(size=n)   # direct Z->Y present

cA, pA = slope(Ya, Zc, Xc2)   # coef on Z given X — should be ~0 if DAG holds
cB, pB = slope(Yb, Zc, Xc2)
print(f'World A: coef(Z | X) = {cA:+.3f}  (p={pA:.2f})  -> Z _||_ Y | X holds, DAG OK')
print(f'World B: coef(Z | X) = {cB:+.3f}  (p={pB:.1e})  -> implication VIOLATED, DAG falsified')
assert abs(cA) < 0.05,        'World A should satisfy Z _||_ Y | X'
assert abs(cB) > 0.3,         'World B should violate the implication'

The same conditional-independence test that *holds* in World A *fails* in World B. That is a DAG earning its keep: it made a prediction, and the data in World B refuted the assumed-missing `Z → Y` edge. (Remember the caveat: passing such tests is necessary, not sufficient — Markov-equivalent DAGs share these implications.)

### 🔧 Exercise 6.1 — derive and test a different implication

In the back-door world from section 5 (`Z → X`, `Z → Y`, `X → Y`), the graph implies **no** unconditional independence between X and Y (they're linked both directly and through Z). But it DOES imply that `Z` and `Y` are *dependent*. As a contrast, build a fresh variable `W` that causes **nothing** (`W ⊥ everything`) and confirm `W ⊥ Y` — a trivially true implication of 'W has no edges.'

Complete the `# TODO`.

In [ ]:
# TODO: W is disconnected from the graph; it should be independent of Y.
W = ...          # TODO: W = RNG.normal(size=n)
c_w = None       # TODO: c_w, p_w = slope(Y, W)
# print(f'coef(W -> Y) = {c_w:+.3f}  (expect ~0)')

### ✅ Solution 6.1

In [ ]:
W = RNG.normal(size=n)
c_w, p_w = slope(Y, W)
print(f'coef(W -> Y) = {c_w:+.3f}  (p={p_w:.2f})  -> W _||_ Y, as a node with no edges must be')
# The honest independence test is the p-value: no evidence of a W->Y link.
assert p_w > 0.01, 'a disconnected node must be independent of Y'
assert abs(c_w) < 0.1, 'the slope should be small (near zero)'

## 7 · Wrap-up & self-check

- A DAG is structural equations you can simulate; **d-separation** predicts which (conditional) independencies hold.
- **Fork** `X←Z→Y` and **chain** `X→M→Y`: conditioning on the middle node BLOCKS the path (`X ⊥ Y | middle`).
- **Collider** `X→K←Y`: blocked by default, and conditioning on K (or a descendant of K) OPENS it — manufacturing association.
- The **back-door criterion** hands you a valid adjustment set; too few controls leave confounding, a collider over-adjusts — only the right set recovers the truth.
- A DAG is **falsifiable**: each missing edge implies a testable conditional independence.

**You're ready for Week 5** if you can classify any triple on sight, read a minimal back-door set off a graph, and name one testable implication of a DAG. Next week: regression turns that adjustment set into an estimate.